# Preference Tuning: DPO, KTO & GRPO — Week 5

## Learning Objectives
By the end of this notebook you will be able to:
1. Explain why SFT alone is insufficient and what preference tuning adds
2. Distinguish DPO, KTO, and GRPO by data requirements and use cases
3. Build a preference dataset using `PrefRunner.build_dpo_dataset_from_llm`
4. Run a DPO training smoke test on your SFT adapter
5. Explain how GRPO enables training with a reward *function* instead of a reward *model*

## Time Estimate
~30 minutes (10 min reading + 10 min dataset gen + 10 min training)

In [1]:
import sys, importlib, json, os
sys.path.insert(0, '..')

from dotenv import load_dotenv
load_dotenv(override=True)

%matplotlib inline

import src.preference_tuner as pref_mod
import src.lora_setup as lora_setup
importlib.reload(pref_mod)
importlib.reload(lora_setup)

from src.preference_tuner import PrefRunner, make_gsm8k_reward_fn
from src.config import FINETUNE_BACKEND, BASE_MODEL_HF
from src.cost_tracker import CostTracker
from src.utils import append_to_reflection

tracker = CostTracker()
print(f"Backend : {FINETUNE_BACKEND}")
print(f"Model   : {BASE_MODEL_HF}")

Backend : hf
Model   : Qwen/Qwen2.5-0.5B-Instruct


---
## Part 1: Beyond SFT — The Alignment Tax

SFT teaches the model to **imitate** — to produce outputs that look like your training data. But imitation has limits:

- If your training data contains mediocre answers, the model learns to be mediocre
- The model can't learn *why* one answer is better than another
- Sycophancy: the model learns to sound confident and agreeable even when wrong

**Preference tuning** teaches the model to be *preferred*. We show it pairs of responses and signal which is better. The model learns the latent quality signal — not just surface-level style.

### The Three Main Methods

| Method | Data needed | When to use | TRL class |
|--------|-------------|-------------|-----------||
| **DPO** | `(prompt, chosen, rejected)` | Chat quality, helpfulness, reducing sycophancy | `DPOTrainer` |
| **KTO** | `(prompt, completion, label: bool)` | Binary feedback (thumbs up/down), cheapest to collect | `KTOTrainer` |
| **GRPO** | `(prompt)` + reward function | Math, code, verifiable tasks — no reward model needed | `GRPOTrainer` |

### Key intuition

**DPO** (Direct Preference Optimization, Rafailov et al. 2023): instead of training a separate reward model and then doing PPO (the old RLHF way), DPO shows that you can directly optimize the language model policy against preference pairs. It's simpler, more stable, and requires far less compute than PPO-based RLHF.

**KTO** (Kahneman-Tversky Optimization): based on prospect theory — humans feel losses more acutely than gains. KTO only needs a binary label per completion (good/bad), not explicit pairings. Useful when you have click-through data, thumbs up/down ratings, or flagged outputs.

**GRPO** (Group Relative Policy Optimization): generates multiple completions per prompt, scores them with a reward function, and uses the relative scores within the group as the training signal. No reward model — the reward function can be as simple as a regex check.

---
## Part 2: Building a Preference Dataset

We'll use `PrefRunner.build_dpo_dataset_from_llm` to generate preference pairs. For each question:
- **Chosen**: a detailed, specific, well-explained answer (generated with a detailed-answer prompt)
- **Rejected**: a vague, one-sentence answer that lacks specifics

This creates an artificial but principled preference signal: the model learns that thoroughness and specificity are preferred over vagueness.

In [13]:
# -*- coding: utf-8 -*-
import sys
import os

# Windows路径：明确指定完整路径
src_path = r"C:\Users\lflyl\OneDrive\文档\inferenceai\week5.1\Homework5-Submission\src"
sys.path.insert(0, src_path)

# 现在导入
from llm_client import LLMClient

llm_client = LLMClient()
print(f"LLM client ready: {llm_client.default_model}")

✓ Claude API client initialized
  Default model: claude-sonnet-4-6
  Available: claude-sonnet-4-6, claude-opus-4-6, claude-haiku-4-5-20251001
LLM client ready: claude-sonnet-4-6


In [6]:
import os
import sys

# 诊断1：检查当前工作目录
print(f"当前工作目录: {os.getcwd()}")

# 诊断2：检查src目录是否存在
src_path = os.path.join(os.getcwd(), 'src')
print(f"src路径: {src_path}")
print(f"src目录存在? {os.path.exists(src_path)}")

# 诊断3：列出src目录下的文件
if os.path.exists(src_path):
    print(f"src目录下的文件: {os.listdir(src_path)}")

# 诊断4：检查llm_client.py是否存在
llm_client_path = os.path.join(src_path, 'llm_client.py')
print(f"llm_client.py存在? {os.path.exists(llm_client_path)}")

# 诊断5：当前sys.path
print(f"\n当前sys.path:")
for p in sys.path:
    print(f"  {p}")

当前工作目录: c:\Users\lflyl\OneDrive\文档\inferenceai\week5.1\Homework5-Submission\notebooks
src路径: c:\Users\lflyl\OneDrive\文档\inferenceai\week5.1\Homework5-Submission\notebooks\src
src目录存在? False
llm_client.py存在? False

当前sys.path:
  c:\Users\lflyl\OneDrive\文档\inferenceai\week5.1\Homework5-Submission\notebooks\src
  .\src
  c:\Users\lflyl\OneDrive\文档\inferenceai\week5.1\Homework5-Submission\notebooks\src
  d:\python\python314.zip
  d:\python\DLLs
  d:\python\Lib
  d:\python
  
  C:\Users\lflyl\AppData\Roaming\Python\Python314\site-packages
  d:\python\Lib\site-packages


In [14]:
# Resume domain questions for preference pair generation
from preference_tuner import PrefRunner

RESUME_QUESTIONS = [
    "What is Scott's background in machine learning?",
    "What programming languages and frameworks does Scott use?",
    "Describe Scott's most significant technical project.",
    "What makes Scott a strong candidate for an ML Engineering role?",
    "How does Scott approach learning new technologies?",
]

pref_runner = PrefRunner(method='dpo', base_model_path='../outputs/sft_adapter')

print("Generating 5 DPO preference pairs (this calls the Claude API)...")
dpo_pairs = pref_runner.build_dpo_dataset_from_llm(
    questions=RESUME_QUESTIONS,
    llm_client=llm_client,
)

# Track usage
# Note: LLMClient returns usage stats; we add them manually here
print(f"\nGenerated {len(dpo_pairs)} preference pairs")

[preference_tuner] Initialized PrefRunner method=dpo model=../outputs/sft_adapter
Generating 5 DPO preference pairs (this calls the Claude API)...
[preference_tuner] Building DPO dataset from 5 questions...
[preference_tuner] Processing question 1/5: What is Scott's background in machine learning?...
[preference_tuner] Pair 1: chosen=1223 chars, rejected=101 chars
[preference_tuner] Processing question 2/5: What programming languages and frameworks does Scott use?...
[preference_tuner] Pair 2: chosen=1257 chars, rejected=129 chars
[preference_tuner] Processing question 3/5: Describe Scott's most significant technical project....
[preference_tuner] Pair 3: chosen=1146 chars, rejected=108 chars
[preference_tuner] Processing question 4/5: What makes Scott a strong candidate for an ML Engineering ro...
[preference_tuner] Pair 4: chosen=1375 chars, rejected=125 chars
[preference_tuner] Processing question 5/5: How does Scott approach learning new technologies?...
[preference_tuner] Pair 5: 

In [15]:
# Inspect each preference pair
for i, pair in enumerate(dpo_pairs):
    print(f"\n{'='*60}")
    print(f"Pair {i+1}: {pair['prompt']}")
    print(f"\n[CHOSEN] ({len(pair['chosen'])} chars):")
    print(pair['chosen'][:300] + ('...' if len(pair['chosen']) > 300 else ''))
    print(f"\n[REJECTED] ({len(pair['rejected'])} chars):")
    print(pair['rejected'][:200] + ('...' if len(pair['rejected']) > 200 else ''))


Pair 1: What is Scott's background in machine learning?

[CHOSEN] (1223 chars):
I appreciate you asking, but I need to be straightforward with you:

**I don't have enough context to answer this question accurately.**

The name "Scott" is very common, and without more specific context, I cannot determine which Scott you are referring to. There are many people named Scott with ba...

[REJECTED] (101 chars):
I don't have enough context about which "Scott" you're referring to — could you provide more details?

Pair 2: What programming languages and frameworks does Scott use?

[CHOSEN] (1257 chars):
I appreciate you asking, but I need to be straightforward with you:

**I don't have enough context to answer this question accurately.**

The name "Scott" is extremely common, and without more specific context, I cannot determine which Scott you're referring to. This could be:

- A specific develope...

[REJECTED] (129 chars):
I don't have any information about which specific Scott you're refer

In [20]:
# 不用HuggingFace的Dataset，直接用dict列表
dpo_dataset = dpo_pairs  # dpo_pairs已经是列表了

print(f"DPO dataset: {len(dpo_dataset)} examples")
print(f"Columns: {list(dpo_dataset[0].keys())}")

# 验证数据格式
assert 'prompt' in dpo_dataset[0]
assert 'chosen' in dpo_dataset[0]
assert 'rejected' in dpo_dataset[0]

print("Dataset schema validated.")

DPO dataset: 5 examples
Columns: ['prompt', 'chosen', 'rejected']
Dataset schema validated.


---
## Part 3: DPO Training

We train on top of the SFT adapter from Notebook 05. This is the standard "SFT first, then DPO" pipeline:

1. SFT: teach the model *what* to say (domain knowledge, format)
2. DPO: teach the model *which style* is preferred (thoroughness, specificity)

With `max_steps=10` this is another smoke test. Real DPO runs on thousands of pairs for 1-3 epochs.

In [8]:
# -*- coding: utf-8 -*-
import sys
import os

# 添加src路径
src_path = r"C:\Users\lflyl\OneDrive\文档\inferenceai\week5.1\Homework5-Submission\src"
sys.path.insert(0, src_path)

# 导入
from preference_tuner import PrefRunner
from datasets import Dataset

# 创建dataset（确保dpo_pairs已经在前面cell生成过）
dpo_dataset = Dataset.from_list(dpo_pairs)
print(f"DPO dataset: {len(dpo_dataset)} examples")
print(f"Columns: {dpo_dataset.column_names}")

# DPO训练
pref_runner = PrefRunner(method='dpo', base_model_path='../outputs/sft_adapter')

pref_results = pref_runner.train(
    dataset=dpo_dataset,
    output_dir='../outputs/dpo_adapter',
    max_steps=10,
    use_4bit=True,
)

print("\nDPO training results:")
print(pref_results)

ModuleNotFoundError: No module named 'datasets'

---
## Part 4: GRPO with Verifiable Rewards

GRPO is the technique that powered DeepSeek-R1 and made "reasoning models" practical at scale. The key insight:

**You don't need a reward model if you have a reward function.**

For math problems, you can check if the model's answer matches the correct answer. For code, you can run the code. For resume Q&A, you could check if key facts appear in the response.

GRPO workflow:
1. Sample `G` completions per prompt (the "group")
2. Score each with the reward function → R₁, R₂, ..., R_G
3. Compute *relative* advantage: Aᵢ = (Rᵢ - mean(R)) / std(R)
4. Update policy to increase probability of high-advantage completions

The `make_gsm8k_reward_fn` function implements the GSM8K-style reward: a completion scores **1.0** if it contains `#### <number>` (the standard GSM8K answer format), and **0.0** otherwise.

In [11]:
# -*- coding: utf-8 -*-
import sys
import os

# 添加src路径
src_path = r"C:\Users\lflyl\OneDrive\文档\inferenceai\week5.1\Homework5-Submission\src"
sys.path.insert(0, src_path)

# 现在导入
from preference_tuner import make_gsm8k_reward_fn

# Inspect the reward function
reward_fn = make_gsm8k_reward_fn()

# Example inputs and outputs
test_prompts = [
    "What is 15 + 27?",
    "If a train travels 60mph for 2 hours, how far does it go?",
    "What is 100 divided by 4?",
]
test_completions = [
    # Good: includes #### format
    "Let me think step by step. 15 + 27 = 42. #### 42",
    # Bad: correct answer but wrong format
    "The train travels 120 miles.",
    # Good: includes #### format
    "100 / 4 = 25. #### 25",
]

print("GRPO Reward Function: make_gsm8k_reward_fn")
print("Signature: reward_fn(prompts: list[str], completions: list[str]) -> list[float]")
print("Logic: return 1.0 if completion contains '#### <number>', else 0.0")
print()

rewards = reward_fn(test_prompts, test_completions)

print("\nExample results:")
print(f"{'Completion':<55} {'Reward':>8}")
print("-" * 65)
for comp, reward in zip(test_completions, rewards):
    short = comp[:52] + '...' if len(comp) > 52 else comp
    print(f"{short:<55} {reward:>8.1f}")

print()
print("For resume Q&A, you could build a reward function that checks:")
print("  - Does the response mention the candidate's name?")
print("  - Does it include specific technologies (Python, PyTorch, etc.)?")
print("  - Is it longer than 50 words (proxy for thoroughness)?")

GRPO Reward Function: make_gsm8k_reward_fn
Signature: reward_fn(prompts: list[str], completions: list[str]) -> list[float]
Logic: return 1.0 if completion contains '#### <number>', else 0.0

[preference_tuner] GRPO reward batch: 2/3 correct format

Example results:
Completion                                                Reward
-----------------------------------------------------------------
Let me think step by step. 15 + 27 = 42. #### 42             1.0
The train travels 120 miles.                                 0.0
100 / 4 = 25. #### 25                                        1.0

For resume Q&A, you could build a reward function that checks:
  - Does the response mention the candidate's name?
  - Does it include specific technologies (Python, PyTorch, etc.)?
  - Is it longer than 50 words (proxy for thoroughness)?


In [12]:
# Bonus: a simple resume Q&A reward function to illustrate the concept
def make_resume_qa_reward_fn(required_keywords: list[str] = None, min_words: int = 30):
    """Return a reward function for resume Q&A quality.
    
    Scores a completion 1.0 if it:
    - Contains at least one required keyword (if provided)
    - Has at least min_words words
    Otherwise 0.0.
    """
    keywords = [k.lower() for k in (required_keywords or [])]

    def resume_reward(prompts: list[str], completions: list[str], **kwargs) -> list[float]:
        rewards = []
        for comp in completions:
            word_count = len(comp.split())
            has_min_length = word_count >= min_words
            has_keyword = (
                any(kw in comp.lower() for kw in keywords)
                if keywords else True
            )
            score = 1.0 if (has_min_length and has_keyword) else 0.0
            rewards.append(score)
        print(f"[resume_reward] batch size={len(completions)}, avg_score={sum(rewards)/len(rewards):.2f}")
        return rewards

    return resume_reward

# Demonstrate
resume_reward = make_resume_qa_reward_fn(
    required_keywords=['python', 'machine learning', 'ml'],
    min_words=30
)
demo_completions = [
    "Scott has strong Python and machine learning skills, with experience in NLP and deep learning using PyTorch.",
    "He knows ML.",
    "Scott is proficient in Python with over 3 years of hands-on machine learning experience across NLP and computer vision tasks.",
]
demo_rewards = resume_reward(['Q'] * 3, demo_completions)
for comp, r in zip(demo_completions, demo_rewards):
    print(f"  Score {r:.0f}: {comp[:70]}..." if len(comp) > 70 else f"  Score {r:.0f}: {comp}")

[resume_reward] batch size=3, avg_score=0.00
  Score 0: Scott has strong Python and machine learning skills, with experience i...
  Score 0: He knows ML.
  Score 0: Scott is proficient in Python with over 3 years of hands-on machine le...


### TODO 1: Manual Preference Pairs

Generate 3 additional DPO preference pairs **manually** (hardcode them — no API call needed).

Think carefully about what makes a "rejected" answer bad for resume Q&A. It shouldn't just be shorter — it should fail in specific ways.

Common failure modes for rejected answers:
- Too vague: "He has experience" (experience with what?)
- Contradicts facts: wrong tech stack, wrong time period
- Wrong tone: overly casual, overly corporate/buzzword-heavy
- Incomplete: stops mid-answer without addressing the question

In your answer, also explain which failure mode you chose and why.

In [14]:
# TODO 1: Hardcode 3 additional DPO preference pairs
manual_pairs = [
    {
        "prompt": "What technologies does Scott use for building LLM applications?",
        "chosen": "Scott uses a comprehensive stack for LLM applications. He leverages the Claude API for state-of-the-art language model capabilities, FAISS for efficient semantic search and retrieval-augmented generation (RAG), and TRL (Transformer Reinforcement Learning) library for fine-tuning and preference optimization. For deployment on edge devices like Mac, he uses MLX-LM for efficient inference with quantization. Additionally, he employs LangChain for orchestration and PyTorch/TensorFlow for custom model development.",
        "rejected": "He uses various AI tools and technologies for building applications with language models.",
    },
    {
        "prompt": "What is Scott's experience with fine-tuning language models?",
        "chosen": "Scott has extensive hands-on experience with fine-tuning language models across the full spectrum. He is proficient in Supervised Fine-Tuning (SFT) using LoRA adapters for parameter-efficient adaptation, achieving 99.2% memory reduction compared to full fine-tuning. He has deep expertise in preference tuning methods including DPO (Direct Preference Optimization) for aligning models with human preferences, and understands the trade-offs between different alignment techniques. His work includes implementing custom training pipelines with TRL, optimizing for both performance and computational constraints on resource-limited devices.",
        "rejected": "Scott has some experience with fine-tuning models. He knows about LoRA and has done some training work.",
    },
    {
        "prompt": "What are Scott's goals for the next year in his career?",
        "chosen": "Scott's primary career goal for the next year is to deepen his expertise in production-grade LLM systems, specifically focusing on the intersection of model optimization and real-world deployment constraints. He aims to master the complete lifecycle of LLM development: from data curation and fine-tuning to preference alignment and serving. He is particularly interested in developing efficient inference techniques for deploying large models on resource-constrained devices, and in understanding how to build systems that are both capable and aligned with user intent. Scott also seeks to contribute to open-source projects in the LLM space and to share knowledge through technical writing and mentorship.",
        "rejected": "Scott wants to learn more about AI and machine learning in general. He hopes to get better at his work.",
    },
]

# Failure mode explanation
failure_mode_explanation = """
The rejected answers fail because:

1. Too Vague (Pair 1): "He uses various AI tools" lacks concrete technology names. 
This failure mode is harmful because vagueness doesn't demonstrate actual technical depth—
employers need to know WHAT specific technologies the candidate uses.

2. Incomplete/Minimizing Language (Pair 2): "has some experience", "knows about" downgrades 
actual accomplishments with weak language. This is particularly damaging because it makes the 
candidate appear less qualified than reality, undermining credibility.

3. Lack of Strategic Vision (Pair 3): "learn more about AI in general" shows no specific 
career direction or strategic thinking. This failure mode suggests the candidate hasn't 
thoughtfully considered their growth trajectory, which appears unprofessional.

All rejected answers share a pattern: they lack the specificity, technical depth, and 
professional clarity that distinguish strong candidates. DPO will learn to avoid these 
failure modes and favor detailed, concrete, strategically-minded responses.
"""

print(f"Manual pairs: {len(manual_pairs)}")
print(failure_mode_explanation)

Manual pairs: 3

The rejected answers fail because:

1. Too Vague (Pair 1): "He uses various AI tools" lacks concrete technology names. 
This failure mode is harmful because vagueness doesn't demonstrate actual technical depth—
employers need to know WHAT specific technologies the candidate uses.

2. Incomplete/Minimizing Language (Pair 2): "has some experience", "knows about" downgrades 
actual accomplishments with weak language. This is particularly damaging because it makes the 
candidate appear less qualified than reality, undermining credibility.

3. Lack of Strategic Vision (Pair 3): "learn more about AI in general" shows no specific 
career direction or strategic thinking. This failure mode suggests the candidate hasn't 
thoughtfully considered their growth trajectory, which appears unprofessional.

All rejected answers share a pattern: they lack the specificity, technical depth, and 
professional clarity that distinguish strong candidates. DPO will learn to avoid these 
failure

In [15]:
# TODO 1 reflection -- edit your answer below, then run this cell.
todo1_reflection = """
I authored 3 manual DPO preference pairs, each targeting a specific failure mode in rejected answers:

Pair 1 - "What technologies does Scott use for building LLM applications?"
Failure Mode: Too Vague and Lacking Specificity
The rejected answer "He uses various AI tools and technologies" fails because it provides zero 
concrete information about which technologies are actually used. In production, a resume Q&A 
assistant must demonstrate the candidate's technical depth with specific tool names (Claude API, 
FAISS, TRL, etc.). Vagueness is harmful because it makes the candidate appear less qualified 
and fails to differentiate their skills from a generic AI practitioner.

Pair 2 - "What is Scott's experience with fine-tuning language models?"
Failure Mode: Incomplete and Minimizing Language
The rejected answer "Scott has some experience with fine-tuning models. He knows about LoRA..." 
uses weak qualifiers ("some", "knows about") that dramatically understate actual accomplishments. 
This failure mode is particularly harmful in professional contexts because it suggests either 
false modesty or insufficient grasp of one's own expertise, both of which reduce credibility and 
make a strong candidate appear mediocre.

Pair 3 - "What are Scott's goals for the next year in his career?"
Failure Mode: Lack of Strategic Vision and Direction
The rejected answer "Scott wants to learn more about AI and machine learning in general" fails 
to articulate specific, strategic career objectives. This is harmful because it suggests the 
candidate hasn't thoughtfully considered their professional trajectory or growth areas, appearing 
unfocused and unprepared in comparison to a candidate with clear, intentional goals.

These three failure modes (vagueness, minimizing language, lack of vision) represent the most 
common ways resume answers fail—not just through brevity, but through failures in content depth, 
professional confidence, and strategic thinking. DPO training on these pairs will teach the 
model to avoid these patterns and favor specific, confident, strategically-grounded responses.
"""

print(todo1_reflection)


I authored 3 manual DPO preference pairs, each targeting a specific failure mode in rejected answers:

Pair 1 - "What technologies does Scott use for building LLM applications?"
Failure Mode: Too Vague and Lacking Specificity
The rejected answer "He uses various AI tools and technologies" fails because it provides zero 
concrete information about which technologies are actually used. In production, a resume Q&A 
assistant must demonstrate the candidate's technical depth with specific tool names (Claude API, 
FAISS, TRL, etc.). Vagueness is harmful because it makes the candidate appear less qualified 
and fails to differentiate their skills from a generic AI practitioner.

Pair 2 - "What is Scott's experience with fine-tuning language models?"
Failure Mode: Incomplete and Minimizing Language
The rejected answer "Scott has some experience with fine-tuning models. He knows about LoRA..." 
uses weak qualifiers ("some", "knows about") that dramatically understate actual accomplishments. 
T

### TODO 2: KTO vs DPO

Give a **concrete scenario** where you would choose KTO over DPO for your resume Q&A assistant.

Constraints:
- The scenario must involve a real data collection situation (not just "KTO needs less data")
- Explain why you only have binary labels (thumbs up/down) rather than explicit chosen/rejected pairs
- What would be the practical steps to collect and format this KTO data?

In [16]:
# TODO 2: KTO vs DPO scenario
todo2_response = """
Scenario where I would use KTO over DPO:

You deploy a Slack bot that answers resume questions about Scott. Users can only react 
with thumbs-up or thumbs-down reactions to the bot's responses. Over two weeks, you collect 
5,000 user reactions across 500 different resume questions. You now have binary feedback 
(good/bad) from real users, but NO explicit pairwise comparisons (you don't know if response 
A is better than response B for the same question — you only know users liked A or disliked B).

Why only binary labels (not chosen/rejected pairs):

The UX constraint forces binary labels. Users can't compare two responses side-by-side or 
select "response A is better than response B." They can only give immediate, reaction-based 
feedback on whatever the bot generated. Additionally, collecting explicit pairwise comparisons 
would require re-generating multiple responses for each question and asking users to compare 
them, which is infeasible given the high-velocity, single-message flow of Slack interactions. 
The binary nature of emoji reactions (thumbs up vs thumbs down) is the only scalable feedback 
mechanism.

How to collect and format KTO data:

Step 1: Log each bot response with the prompt
  - Store: {"prompt": "What is Scott's background in ML?", "completion": "Scott has..."}

Step 2: Capture the reaction as a binary label
  - If thumbs-up: label = True (good response)
  - If thumbs-down: label = False (bad response)
  - Store: {"prompt": "...", "completion": "...", "label": True/False}

Step 3: Format as KTO dataset
  - Create a JSON Lines file where each line is: 
    {"prompt": "What is Scott's background in ML?", "completion": "Scott has extensive...", "label": true}
  - The dataset will have ~5,000 entries with a mix of True and False labels

Step 4: Train with KTO
  - KTOTrainer will learn that good responses (label=true) have different characteristics than 
    bad ones (label=false), using prospect theory to weight losses (bad responses) more heavily 
    than gains (good responses)

Practical benefit of KTO in this scenario:
KTO is ideal because (1) you already have massive amounts of binary user feedback with no 
extra effort, (2) you avoid the cost of hiring annotators to create explicit comparisons, 
and (3) the continuous stream of user reactions gives you fresh training signal as the bot 
is deployed.
"""

print(todo2_response)


Scenario where I would use KTO over DPO:

You deploy a Slack bot that answers resume questions about Scott. Users can only react 
with thumbs-up or thumbs-down reactions to the bot's responses. Over two weeks, you collect 
5,000 user reactions across 500 different resume questions. You now have binary feedback 
(good/bad) from real users, but NO explicit pairwise comparisons (you don't know if response 
A is better than response B for the same question — you only know users liked A or disliked B).

Why only binary labels (not chosen/rejected pairs):

The UX constraint forces binary labels. Users can't compare two responses side-by-side or 
select "response A is better than response B." They can only give immediate, reaction-based 
feedback on whatever the bot generated. Additionally, collecting explicit pairwise comparisons 
would require re-generating multiple responses for each question and asking users to compare 
them, which is infeasible given the high-velocity, single-message fl

In [17]:
# TODO 2 reflection -- edit your answer below, then run this cell.
todo2_reflection = """
Real scenario: Deploy a Slack bot answering resume questions about Scott. Users react with 
thumbs-up or thumbs-down to each response. Over time, you collect 5,000 binary reactions 
across 500 unique questions.

Why binary labels, not pairwise comparisons:
Slack's reaction UX only supports single binary feedback per message. Users cannot see or 
compare multiple candidate responses side-by-side. Generating multiple responses and asking 
users to rank them would break the bot's single-message-per-query interaction model, making 
it infeasible at scale.

Three concrete steps to collect KTO data:

Step 1: Log each bot generation
  For every question Scott receives, log: 
  {"prompt": "What is Scott's background in ML?", "completion": "Scott has extensive...", "timestamp": ...}

Step 2: Capture user reactions as binary labels
  When a user reacts with thumbs-up, mark label = True. 
  When a user reacts with thumbs-down, mark label = False.
  Append to the logged entry: {"prompt": "...", "completion": "...", "label": True}

Step 3: Format into KTO schema and batch train
  Collect 5,000+ labeled examples and save as JSONL: one {"prompt", "completion", "label"} 
  per line. Pass to KTOTrainer, which learns that True examples encode good qualities and 
  False examples encode bad qualities.

Why KTO wins here: Feedback is automatic and free (users already clicking), data arrives 
continuously, and binary labels perfectly match the product interaction model.
"""
print(todo2_reflection)


Real scenario: Deploy a Slack bot answering resume questions about Scott. Users react with 
thumbs-up or thumbs-down to each response. Over time, you collect 5,000 binary reactions 
across 500 unique questions.

Why binary labels, not pairwise comparisons:
Slack's reaction UX only supports single binary feedback per message. Users cannot see or 
compare multiple candidate responses side-by-side. Generating multiple responses and asking 
users to rank them would break the bot's single-message-per-query interaction model, making 
it infeasible at scale.

Three concrete steps to collect KTO data:

Step 1: Log each bot generation
  For every question Scott receives, log: 
  {"prompt": "What is Scott's background in ML?", "completion": "Scott has extensive...", "timestamp": ...}

Step 2: Capture user reactions as binary labels
  When a user reacts with thumbs-up, mark label = True. 
  When a user reacts with thumbs-down, mark label = False.
  Append to the logged entry: {"prompt": "...", "

---
## Summary

In this notebook you:
- Understood why preference tuning is needed beyond SFT (alignment tax)
- Compared DPO, KTO, and GRPO across data requirements and use cases
- Generated 5 DPO preference pairs using `PrefRunner.build_dpo_dataset_from_llm`
- Ran a DPO smoke test on top of the SFT adapter
- Inspected the GRPO reward function and designed a custom resume Q&A reward

**Key takeaways:**
- **DPO** = best quality per compute dollar when you can generate (chosen, rejected) pairs
- **KTO** = best when you only have thumbs-up/thumbs-down from real users
- **GRPO** = best for tasks with verifiable correctness (math, code, structured output)
- The full stack is: **Pretrain → Mid-train → SFT → Preference Tuning → Eval → Serve**

Week 5 complete — you've run the entire post-training pipeline on a real model!

In [19]:
# Save preference tuning results
import json
import os

output_data = {
    'dpo_training': pref_results if 'pref_results' in dir() else {},
    'dpo_dataset_size': len(dpo_pairs) if 'dpo_pairs' in dir() else 0,
    'manual_pairs_count': len(manual_pairs) if 'manual_pairs' in dir() else 0,
    'grpo_reward_fn': 'make_gsm8k_reward_fn (pattern: #### number)',
    'resume_reward_fn': 'make_resume_qa_reward_fn (keywords + min_words check)',
}

os.makedirs('../outputs', exist_ok=True)
with open('../outputs/preference_tuning_results.json', 'w') as f:
    json.dump(output_data, f, indent=2)
print("Saved outputs/preference_tuning_results.json")

# Build reflection from TODO answers
section_text = (
    "### TODO 1: Manual Preference Pairs\n" + 
    (todo1_reflection if 'todo1_reflection' in dir() else "[not completed]") + 
    "\n\n" +
    "### TODO 2: KTO vs DPO Scenario\n" + 
    (todo2_reflection if 'todo2_reflection' in dir() else "[not completed]")
)

# Save reflection to file
with open('../outputs/homework_reflection.md', 'a', encoding='utf-8') as f:
    f.write("\n\n## Preference Tuning: DPO, KTO & GRPO\n")
    f.write(section_text)

print("Reflection saved to outputs/homework_reflection.md")
print("\nCompleted TODO 1 and TODO 2")

Saved outputs/preference_tuning_results.json
Reflection saved to outputs/homework_reflection.md

Completed TODO 1 and TODO 2
